In [0]:
from pyspark.sql.functions import col,when,lit,from_json,sum as s
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,LongType,BinaryType,TimestampType,DoubleType,FloatType,BooleanType
from datetime import datetime
from zoneinfo import ZoneInfo

In [0]:
storage = "abfss://onlinestore@onlinestorage2026.dfs.core.windows.net/"

bronze_versions = [f.name for f in dbutils.fs.ls(storage + "bronze") if f.isDir()]
latest_version = sorted(bronze_versions)[-1]

bronze_path = storage + "bronze/" + latest_version
print(f"Latest version: {latest_version}")
print(f"Bronze path: {bronze_path}")

In [0]:
bronze = spark.read.format("delta").load(bronze_path)
display(bronze)

In [0]:
bronze_df = (spark.readStream
              .format("delta").load(bronze_path)
              )

In [0]:
unified_schema = StructType([
    # Common fields
    StructField("source", StringType(), True),
    StructField("event_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("month", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("gross_sales", DoubleType(), True),
    StructField("discounts", DoubleType(), True),
    StructField("returns", DoubleType(), True),
    # monthly_sales fields
    StructField("total_orders", IntegerType(), True),
    StructField("net_sales", DoubleType(), True),
    StructField("shipping", DoubleType(), True),
    StructField("total_sales", DoubleType(), True),
    # product_sales fields
    StructField("product_id", StringType(), True),
    StructField("product_type", StringType(), True),
    StructField("net_quantity", IntegerType(), True),
    StructField("total_net_sales", DoubleType(), True),
])

In [0]:
# Streaming version for the pipeline
prase_df = bronze_df.withColumn("data", from_json(col("raw_json"), unified_schema)).select("data.*")

# Batch version for analysis/testing
df = bronze.withColumn("data", from_json(col("raw_json"), unified_schema)).select("data.*")
display(df)

In [0]:
check_null = df.select(*[
    s(col(c).isNull().cast("int")).alias(c)
    for c in df.columns

])
display(check_null)

In [0]:
# Split by source and keep only relevant columns (no nulls)
product_sales_df = df.filter(col("source") == "product_sales").select(
    "event_id", "event_timestamp", "product_id", "product_type", "month", "year",
    "net_quantity", "gross_sales", "discounts", "returns", "total_net_sales"
)

monthly_sales_df = df.filter(col("source") == "monthly_sales").select(
    "event_id", "event_timestamp", "month", "year",
    "total_orders", "gross_sales", "discounts", "returns", "net_sales", "shipping", "total_sales"
)

print(f"Product sales rows: {product_sales_df.count()}")
print(f"Monthly sales rows: {monthly_sales_df.count()}")
display(product_sales_df)

In [0]:
check_null_1 = product_sales_df.select(
    *[
        s(col(c).isNull().cast("int")).alias(c)
        for c in product_sales_df.columns
    ]
)
display(check_null_1)

In [0]:
check_null_2 = monthly_sales_df.select(
    *[
        s(col(c).isNull().cast("int")).alias(c)
        for c in monthly_sales_df.columns
    ]
)
display(check_null_2)

In [0]:
product_sales_df.printSchema()
monthly_sales_df.printSchema()

In [0]:
sliver_path = storage +f"sliver"
PIPELINE_VERSION = "V-"+datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%Y-%m-%d-%H-%M-%S")
product_sliver_path = sliver_path + f"/product_sales/{PIPELINE_VERSION}"
monthly_sliver_path = sliver_path + f"/monthly_sales/{PIPELINE_VERSION}"
sliver_cpkt_product =  product_sliver_path+"/checkpoint"
sliver_cpkt_mothly =  monthly_sliver_path+"/checkpoint"
print(f"Product sliver path: {sliver_cpkt_product}")
print(f"Monthly sliver path: {sliver_cpkt_mothly}")

In [0]:
# Streaming product sales to silver storage (no nulls)
product_stream = (
    prase_df.filter(col("source") == "product_sales")
    .select("event_id", "event_timestamp", "product_id", "product_type", "month", "year",
            "net_quantity", "gross_sales", "discounts", "returns", "total_net_sales")
    .na.drop()
    .writeStream
    .format("delta")
    .option("checkpointLocation", sliver_cpkt_product)
    .outputMode("append")
    .option("mergeSchema", True)
    .option("overwriteSchema", True)
    .start(product_sliver_path)
)

In [0]:
# Streaming monthly sales to silver storage (no nulls)
monthly_stream = (
    prase_df.filter(col("source") == "monthly_sales")
    .select("event_id", "event_timestamp", "month", "year",
            "total_orders", "gross_sales", "discounts", "returns", "net_sales", "shipping", "total_sales")
    .na.drop()
    .writeStream
    .format("delta")
    .option("checkpointLocation", sliver_cpkt_mothly)
    .outputMode("append")
    .option("mergeSchema", True)
    .option("overwriteSchema", True)
    .start(monthly_sliver_path)
)


In [0]:
product_df = spark.read.format("delta").load(product_sliver_path)
display(product_df)

In [0]:
monthly_df = spark.read.format("delta").load(monthly_sliver_path)
display(monthly_df)